# Malinois MPRA: allelic effects in HepG2 and K562 vs the function CAV

The Kircher MPRA has only 4 regions in the cells the CAV was trained on (HepG2, K562), so the pooled evaluation
rests on 4 regions. Gosai et al. 2024 (Nature, "Machine-guided design of cell-type-targeting cis-regulatory
elements") released the MPRA data used to train Malinois: **Supplementary Table 2**, ~800k 200 bp sequences with
log2FC and its standard error in **K562, HepG2 and SK-N-SH**. Most are the reference (R) and alternative (A) allele of
fine-mapped UK Biobank and GTEx variants, the variant at base 100 of the 200 (checked: index 99, coordinates hg19).

**Question.** For a variant inside a motif, is its measured allelic effect (log2FC of A − log2FC of R) larger when the
CAV says the motif is bound? Does it help motifbreakR predict the effect? Does this hold more in HepG2/K562 (the CAV's
training cells) than in SK-N-SH (not)? The same sequences are measured in all three cells, so SK-N-SH is a clean
control: only the cell changes.

**What is reused unchanged.** The NT-v2 500M function directions of the pooled-regions run (`fcav_pool/fits/nt500m/`,
251 TFs, layer 24; no refit), the readout (600 bp window, motif at bp 300, mean over the tokens touching the motif),
FIMO hits at `fimo_p`, motifbreakR's scoring and thresholds, and the rule motifbreakR + CAV = alleleDiff × w(P),
w = sigmoid(20 (P − 0.75)).

**What is new.**
* P(bound) of a new occurrence uses the direction fitted on **all** sites of the TF (`w_all`, `mu`, `sd` in the fit
  file), since these occurrences are not in any fold. 2.2 checks it against the out-of-fold `prob_hit` of the pooled hits.
* The readout window is hg38 around the motif (the 200 bp MPRA sequence matches hg38 exactly after liftover; checked
  per variant, mismatches dropped), as for the Kircher regions. The extra 400 bp were not in the reporter.
* **Leakage.** An occurrence whose 600 bp window overlaps a ChIP site of the **same TF** used to fit its direction is
  dropped (counted in 1.5).
* No regions here, so the null is N2 only: P shuffled among the occurrences of the same TF. It keeps which TFs are
  called and asks whether P adds anything within a TF.

**Run order.** Section 1 on a **CPU** runtime (download, liftover, FIMO, motifbreakR; well under an hour). Section 2
on a **GPU** runtime (one forward pass per occurrence, resumable per TF). Section 3 (evaluation, a few minutes) right
after Section 2 in the same session, or later on a CPU runtime.

Output on Drive: `motif_token_study/malinois/`.

---
# Section 1 — variants, hits, motifbreakR *(CPU runtime)*

In [ ]:
!pip install -q pymemesuite pyfaidx pyliftover
from google.colab import drive
drive.mount('/content/drive')

## 1.1 Supplementary Table 2 → one row per SNV with both alleles

Kept: UKBB and GTEx rows (not the CRE set), centred windows (`wC`), SNVs, 200 bp. A variant tested in more than one
library keeps its first copy (~1%). Per cell: Δ = log2FC(A) − log2FC(R), its SE = √(SE_A² + SE_R²), and the activity
of the element = max(log2FC(A), log2FC(R)).

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time, shutil, subprocess
import numpy as np, pandas as pd

C = json.load(open(f"{WORK_DIR}/config.json"))
POOL = f"{WORK_DIR}/fcav_pool"
MAL = f"{WORK_DIR}/malinois"
os.makedirs(MAL, exist_ok=True)
TABLE_URL = ("https://static-content.springer.com/esm/art%3A10.1038%2Fs41586-024-08070-z/MediaObjects/"
             "41586_2024_8070_MOESM4_ESM.txt")
TABLE = f"{MAL}/gosai2024_supp_table2.txt"
CELLS3 = ["HepG2", "K562", "SKNSH"]
VPOS = 99                      # the variant's index in the 200 bp sequence
MOFF, WLEN = C["motif_offset"], C["window_len"]
pd.set_option("display.width", 250)

if not os.path.exists(TABLE):
    subprocess.run(["wget", "-q", "-O", TABLE + ".part", TABLE_URL], check=True)
    os.replace(TABLE + ".part", TABLE)
d = pd.read_csv(TABLE, sep="\t", dtype={"chr": str, "OL": str}, low_memory=False)
print(f"{len(d):,} rows;", d["data_project"].value_counts().to_dict())
d = d[d["data_project"] != "CRE"]
p = d["IDs"].str.split(":", expand=True)
d = d.assign(chrom=p[0], pos=pd.to_numeric(p[1]), ref=p[2], alt=p[3], allele=p[4], win=p[5])
d = d[(d["win"] == "wC") & (d["ref"].str.len() == 1) & (d["alt"].str.len() == 1) & (d["sequence"].str.len() == 200)]
d["var"] = d["chrom"] + ":" + d["pos"].astype(str) + ":" + d["ref"] + ":" + d["alt"]
d = d.drop_duplicates(["var", "allele"])
cols = [f"{c}_log2FC" for c in CELLS3] + [f"{c}_lfcSE" for c in CELLS3]
R_, A_ = d[d["allele"] == "R"].set_index("var"), d[d["allele"] == "A"].set_index("var")
V = (R_[["chrom", "pos", "ref", "alt", "data_project", "class", "sequence"]]
     .join(R_[cols].add_prefix("R_")).join(A_[cols + ["sequence"]].add_prefix("A_"), how="inner"))
s, a = V["sequence"], V["A_sequence"]
ok = ((s.str[VPOS] == V["ref"]) & (a.str[VPOS] == V["alt"]) & (s.str[:VPOS] == a.str[:VPOS])
      & (s.str[VPOS + 1:] == a.str[VPOS + 1:]) & ~s.str.contains("N"))
print(f"{len(V):,} SNVs with both alleles; {int((~ok).sum())} dropped (alleles not at base {VPOS + 1}, or N)")
V = V[ok].drop(columns="A_sequence").reset_index()
for c in CELLS3:
    V[f"d_{c}"] = V[f"A_{c}_log2FC"] - V[f"R_{c}_log2FC"]
    V[f"se_{c}"] = np.sqrt(V[f"A_{c}_lfcSE"] ** 2 + V[f"R_{c}_lfcSE"] ** 2)
    V[f"act_{c}"] = np.maximum(V[f"A_{c}_log2FC"], V[f"R_{c}_log2FC"])
print(V["data_project"].value_counts().to_string())
print(pd.DataFrame({c: {"|Δ| median": V[f"d_{c}"].abs().median(), "|Δ/SE| >= 2": (V[f"d_{c}"].abs() / V[f"se_{c}"] >= 2).mean(),
                        "active (log2FC >= 1)": (V[f"act_{c}"] >= 1).mean()} for c in CELLS3}).round(3).to_string())

## 1.2 hg19 → hg38, and the hg38 bases must equal the MPRA sequence

In [ ]:
from pyliftover import LiftOver
from pyfaidx import Fasta

LOCAL_FA = "/content/hg38.fa"
if not os.path.exists(LOCAL_FA):
    print("copying hg38 to local disk (~3 GB, one-off)...")
    shutil.copy(C["genome_fa"], LOCAL_FA)
    if os.path.exists(C["genome_fa"] + ".fai"):
        shutil.copy(C["genome_fa"] + ".fai", LOCAL_FA + ".fai")
GENOME = Fasta(LOCAL_FA, sequence_always_upper=True)
CHROM_LEN = {c: len(GENOME[c]) for c in GENOME.keys()}
LO = LiftOver("hg19", "hg38")
c38, p38 = [], []
for ch, ps in zip(V["chrom"], V["pos"]):
    r = LO.convert_coordinate(f"chr{ch}", int(ps) - 1)
    if r and r[0][2] == "+" and r[0][0] in CHROM_LEN:
        c38.append(r[0][0]); p38.append(int(r[0][1]))
    else:
        c38.append(None); p38.append(-1)
V["chrom38"], V["pos38"] = c38, p38                 # 0-based hg38 position of the variant
same = np.array([c is not None and q >= VPOS and GENOME[c][q - VPOS:q - VPOS + 200].seq == s
                 for c, q, s in zip(V["chrom38"], V["pos38"], V["sequence"])])
print(f"lifted and identical to hg38: {same.sum():,} of {len(V):,}; dropped {int((~same).sum()):,}")
V = V[same].reset_index(drop=True)

## 1.3 FIMO hits that contain the variant, for the 251 TFs with a direction

Only the bases a motif containing base 100 can reach are scanned (FIMO p-values do not depend on the rest of the
sequence). An occurrence is identified by its hg38 position, strand and TF, so neighbouring variants that sit in the
same occurrence share one P(bound).

In [ ]:
from pymemesuite.common import Sequence, MotifFile
from pymemesuite.fimo import FIMO

with MotifFile(C["meme_path"]) as _mf:
    _all = list(_mf)
    BACKGROUND = _mf.background
MOTIF = {}
for m in _all:
    MOTIF.setdefault(m.name.decode(), m)           # first matrix with the name, as everywhere else
FIMO_SCAN = FIMO(both_strands=True, threshold=C["fimo_p"])
FIT_DIR = f"{POOL}/fits/nt500m"
TFS = sorted(f[:-4] for f in os.listdir(FIT_DIR) if f.endswith(".npz") and not f.endswith(".part.npz"))
TFS = [m for m in TFS if m in MOTIF]
WID = {m: np.asarray(MOTIF[m].frequencies).shape[0] for m in TFS}
MAXW = max(WID.values())
LO_, HI_ = VPOS - MAXW + 1, VPOS + MAXW
CHUNKS = []
_sub = [s[LO_:HI_] for s in V["sequence"]]
for i0 in range(0, len(_sub), 5000):
    CHUNKS.append([Sequence(x, name=str(i0 + k).encode()) for k, x in enumerate(_sub[i0:i0 + 5000])])
del _sub
t0, rows = time.time(), []
for k, m in enumerate(TFS):
    for ch in CHUNKS:
        for h in FIMO_SCAN.score_motif(MOTIF[m], ch, BACKGROUND).matched_elements:
            a, b = sorted((h.start, h.stop))
            a, b = a - 1 + LO_, b + LO_
            if a <= VPOS < b:
                rows.append((int(h.source.name.decode()), m, a, b, h.strand, h.pvalue))
    if k %% 25 == 0:
        print(f"  {k + 1}/{len(TFS)} TFs, {len(rows):,} hits [{time.time() - t0:.0f}s]", flush=True)
H = pd.DataFrame(rows, columns=["vi", "motif", "a", "b", "strand", "p"])
assert (H["b"] - H["a"] == H["motif"].map(WID)).all()
H["chrom38"] = V["chrom38"].values[H["vi"]]
H["g38"] = V["pos38"].values[H["vi"]] - VPOS + H["a"]          # hg38 start of the motif
inside = (H["g38"] - MOFF >= 0) & (H["g38"] - MOFF + WLEN <= H["chrom38"].map(CHROM_LEN))
H = H[inside].reset_index(drop=True)
H["uid"] = pd.factorize(H["chrom38"] + ":" + H["g38"].astype(str) + ":" + H["strand"] + ":" + H["motif"])[0]
print(f"{len(H):,} (variant, hit) pairs; {H['vi'].nunique():,} variants in at least one hit; "
      f"{H['uid'].nunique():,} distinct occurrences; {H['motif'].nunique()} TFs  [{(time.time() - t0) / 60:.1f} min]")

## 1.4 motifbreakR's alleleDiff for every (variant, TF) with a hit

The scoring of the pooled evaluation notebook (`build`): best window over both strands among those containing the
variant, scored with the information-weighted matrix; a call needs a relative score > 0.85 for one allele and
|alleleDiff| ≥ 0.4. A call counts only for a TF that also has a FIMO hit containing the variant, as there.

In [ ]:
import re

def _read_meme(path):
    out, name, rows_, ns = {}, None, [], 20
    for line in open(path):
        t = line.split()
        if line.startswith("MOTIF"):
            if name is not None and name not in out:
                out[name] = (np.array(rows_), ns)
            name, rows_, ns = (t[2] if len(t) > 2 else t[1]), [], 20
        elif line.startswith("letter-probability"):
            mm = re.search(r"nsites=\s*([0-9.]+)", line)
            ns = float(mm.group(1)) if mm else 20
        elif name is not None and len(t) == 4 and all(re.fullmatch(r"[0-9.eE+-]+", x) for x in t):
            rows_.append([float(x) for x in t])
    if name is not None and name not in out:
        out[name] = (np.array(rows_), ns)
    return out


MBR_THRESH, MBR_NEUT = 0.85, 0.4
MBR_MATS = {}
for name, (f, ns) in _read_meme(C["meme_path"]).items():
    f = f / f.sum(1, keepdims=True)
    pp = (f * ns + 0.25) / (ns + 1)
    S = pp * (pp.max(1) - pp.min(1))[:, None]
    S = np.vstack([S.T, np.zeros(len(S))]).T
    MBR_MATS[name] = (S, S[::-1][:, [3, 2, 1, 0, 4]], S[:, :4].min(1).sum(), S[:, :4].max(1).sum())
ENC = np.full(256, 4, np.int8)
for i, b in enumerate("ACGT"):
    ENC[ord(b)] = i
SEQ = ENC[np.frombuffer("".join(V["sequence"]).encode(), np.uint8)].reshape(len(V), 200)
REF, ALT = SEQ[:, VPOS], ENC[np.frombuffer("".join(V["alt"]).encode(), np.uint8)]
calls = []
for m, g in H[["vi", "motif"]].drop_duplicates().groupby("motif"):
    if m not in MBR_MATS:
        continue
    S, Sr, smin, smax = MBR_MATS[m]
    w = len(S)
    st = np.arange(VPOS - w + 1, VPOS + 1)
    st = st[(st >= 0) & (st + w <= 200)]
    vi = g["vi"].values
    X = SEQ[vi][:, st[:, None] + np.arange(w)[None, :]]                # (n, windows, w)
    k = np.arange(w)[None, None, :]
    rf, rr = S[k, X].sum(2), Sr[k, X].sum(2)
    kp = (VPOS - st)[None, :]
    rb, ab = REF[vi][:, None], ALT[vi][:, None]
    af, ar = rf + S[kp, ab] - S[kp, rb], rr + Sr[kp, ab] - Sr[kp, rb]
    sr, sa = np.maximum(rf, rr).max(1), np.maximum(af, ar).max(1)
    pr, pa = (sr - smin) / (smax - smin), (sa - smin) / (smax - smin)
    diff = sa - sr
    keep = ((pr > MBR_THRESH) | (pa > MBR_THRESH)) & (np.abs(diff) >= MBR_NEUT)
    calls.append(pd.DataFrame({"vi": vi[keep], "motif": m, "alleleDiff": diff[keep]}))
CALLS = pd.concat(calls, ignore_index=True)
print(f"{len(CALLS):,} motifbreakR calls on {CALLS['vi'].nunique():,} variants, {CALLS['motif'].nunique()} TFs")

## 1.5 Leakage: occurrences that sit on a ChIP site of the same TF, then save

In [ ]:
H["leak"] = False
for m, g in H.groupby("motif"):
    J = json.load(open(f"{POOL}/sites/{m}.json"))
    st = pd.DataFrame([(r["chrom"], r["g_start"]) for r in J["records"]], columns=["chrom", "g"])
    for ch, gg in g.groupby("chrom38"):
        s = np.sort(st.loc[st["chrom"] == ch, "g"].values)
        if not len(s):
            continue
        x = gg["g38"].values
        i = np.searchsorted(s, x)
        near = np.minimum(np.abs(x - s[np.clip(i - 1, 0, len(s) - 1)]), np.abs(x - s[np.clip(i, 0, len(s) - 1)]))
        H.loc[gg.index, "leak"] = near < WLEN
u = H.drop_duplicates("uid")
print(f"occurrences overlapping a fitted site of the same TF: {int(u['leak'].sum()):,} of {len(u):,} "
      f"({u['leak'].mean():.1%}); they get no P(bound)")
V.drop(columns="sequence").to_csv(f"{MAL}/variants.csv.gz", index_label="vi")
H.to_csv(f"{MAL}/hits.csv", index=False)
CALLS.to_csv(f"{MAL}/calls.csv", index=False)
print(f"saved to {MAL}: {len(V):,} variants, {len(H):,} hit rows, {len(CALLS):,} calls")

In [ ]:
from google.colab import runtime
runtime.unassign()

---
# Section 2 — P(bound) of every occurrence, NT-v2 500M L24 *(GPU runtime)*

## 2.1 Settings, and the readout copied from `Pooled Regions 1c`

In [ ]:
!pip install -q -U "transformers==4.57.6" "tokenizers==0.22.0" "huggingface_hub==0.34.0" "safetensors==0.4.5"
!pip install -q scikit-learn scipy pandas pyfaidx
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json, time, gc, sys, shutil
import numpy as np, pandas as pd
import torch
from scipy import stats
from pyfaidx import Fasta
from transformers import AutoTokenizer, AutoModel, AutoConfig, AutoModelForMaskedLM, AutoModelForCausalLM

C = json.load(open(f"{WORK_DIR}/config.json"))
POOL = f"{WORK_DIR}/fcav_pool"
MAL = f"{WORK_DIR}/malinois"
MWJ = json.load(open(f"{POOL}/mpra_windows.json"))
TAG, LAYER = "nt500m", 24
REGISTRY = {"nt500m": dict(hf="InstaDeepAI/nucleotide-transformer-v2-500m-multi-species", tok_map=("kmer", 6),
                           causal=False, batch=32)}
FIT_LAYERS = {"nt500m": [LAYER]}
BASE_WEIGHTED = False
BATCH = 32
WLEN, MOFF = C["window_len"], C["motif_offset"]
DEV = torch.device("cuda")
COMP = str.maketrans("ACGTN", "TGCAN")
rc = lambda s: s.translate(COMP)[::-1]
FIT_DIR = f"{POOL}/fits/{TAG}"
OUTP = f"{MAL}/p_{TAG}"
os.makedirs(OUTP, exist_ok=True)

LOCAL_FA = "/content/hg38.fa"
if not os.path.exists(LOCAL_FA):
    print("copying hg38 to local disk (~3 GB, one-off)...")
    shutil.copy(C["genome_fa"], LOCAL_FA)
    if os.path.exists(C["genome_fa"] + ".fai"):
        shutil.copy(C["genome_fa"] + ".fai", LOCAL_FA + ".fai")
GENOME = Fasta(LOCAL_FA, sequence_always_upper=True)
H = pd.read_csv(f"{MAL}/hits.csv")
U = H[~H["leak"]].drop_duplicates("uid")
print(f"{len(U):,} occurrences to embed, {U['motif'].nunique()} TFs")

In [ ]:
def load_tokenizer(tag):
    spec = REGISTRY[tag]
    tokz = AutoTokenizer.from_pretrained(spec["hf"], trust_remote_code=True)
    tokz.padding_side = "right"          # several of these default to left (HyenaDNA)
    if tokz.pad_token is None:
        tokz.pad_token = tokz.eos_token if tokz.eos_token is not None else tokz.unk_token
    return tokz


def token_bases(spec, n):
    # (n, 2) base interval of each sequence token, for the arithmetic maps
    kind = spec["tok_map"][0]
    if kind == "kmer":
        k = spec["tok_map"][1]
        return np.stack([np.arange(n) * k, (np.arange(n) + 1) * k], 1)
    if kind == "char":
        return np.stack([np.arange(n), np.arange(n) + 1], 1)
    raise RuntimeError("offsets are resolved by the tokenizer, not here")


def encode(tokz, spec, seqs):
    # input_ids (B, T), attention mask (B, T), and per sequence: the positions of its sequence tokens in input_ids
    # and the base interval each one covers
    kind = spec["tok_map"][0]
    if kind == "offsets":
        assert tokz.is_fast, f"{spec['hf']} has no fast tokenizer, so offset mapping is unavailable"
        enc = tokz(seqs, return_tensors="pt", padding=True, return_offsets_mapping=True)
    else:
        enc = tokz(seqs, return_tensors="pt", padding=True)
    ids = enc["input_ids"]
    am = enc.get("attention_mask")
    if am is None:
        am = (ids != tokz.pad_token_id).long()
    amn = am.numpy()
    pos, spans = [], []
    if kind == "offsets":
        om = enc["offset_mapping"].numpy()
        for b in range(len(seqs)):
            keep = (om[b, :, 1] > om[b, :, 0]) & (amn[b] == 1)
            pos.append(np.flatnonzero(keep))
            spans.append(om[b, keep])
    else:
        for b in range(len(seqs)):
            sp = np.array(tokz.get_special_tokens_mask(ids[b].tolist(), already_has_special_tokens=True))
            p = np.flatnonzero((sp == 0) & (amn[b] == 1))
            pos.append(p)
            spans.append(token_bases(spec, len(p)))
        assert len({len(p) for p in pos}) == 1, f"{spec['hf']}: windows of equal length gave different token counts"
    return ids, am, pos, spans


def span_weights(bases, lo, hi):
    # per-token weight for the base interval [lo, hi)
    ov = np.minimum(bases[:, 1], hi) - np.maximum(bases[:, 0], lo)
    ov = np.clip(ov, 0, None).astype(np.float32)
    if not BASE_WEIGHTED:
        ov = (ov > 0).astype(np.float32)
    assert ov.sum() > 0, f"no token covers bases [{lo}, {hi})"
    return ov / ov.sum()


def load_model(tag):
    spec = REGISTRY[tag]
    # the class the checkpoint ships its own code for (NT-v2 has AutoModelForMaskedLM but no AutoModel)
    amap = getattr(AutoConfig.from_pretrained(spec["hf"], trust_remote_code=True), "auto_map", None) or {}
    cls = next((c for k, c in (("AutoModel", AutoModel), ("AutoModelForMaskedLM", AutoModelForMaskedLM),
                               ("AutoModelForCausalLM", AutoModelForCausalLM)) if k in amap), AutoModel)
    mdl = cls.from_pretrained(spec["hf"], trust_remote_code=True).to(DEV).eval()
    # DNABERT-2's triton flash-attention kernel does not compile on current triton; its code falls back to PyTorch
    for name in {type(m).__module__ for m in mdl.modules()}:
        mod = sys.modules.get(name)
        if mod is not None and getattr(mod, "flash_attn_qkvpacked_func", None) is not None:
            mod.flash_attn_qkvpacked_func = None
            print(f"{tag}: triton flash-attention disabled in {name.split('.')[-1]}; using PyTorch attention")
    if spec.get("out") == "dnabert2" and getattr(mdl, "pooler", None) is not None:
        mdl.pooler = None
    return mdl


POS, DEPTH, LOADED = {}, {}, {}


@torch.no_grad()
def hidden_stack(tag, model, ids, am):
    # one (layers, T_j, H) float tensor per sequence; only the layers in POS[tag] once prepare() has set it
    spec = REGISTRY[tag]
    if spec.get("out") == "dnabert2":
        # DNABERT-2 returns every encoder layer unpadded; one sequence at a time keeps the token order
        out = []
        for j in range(ids.shape[0]):
            t = ids[j][am[j].bool()][None].to(DEV)
            layers, _ = model(t, attention_mask=torch.ones_like(t), output_all_encoded_layers=True)
            S = torch.stack([x.reshape(t.shape[1], -1) for x in layers], 0).float()
            out.append(S[POS[tag]] if tag in POS else S)
        return out
    kw = {} if spec["causal"] else {"attention_mask": am.to(DEV)}
    hs = model(input_ids=ids.to(DEV), output_hidden_states=True, **kw).hidden_states
    if tag in POS:
        hs = [hs[i] for i in POS[tag]]
    S = torch.stack(hs, 1).float()
    return [S[j] for j in range(S.shape[0])]


PROBE = [h["window"] for h in MWJ["hits"][:4]]


def prepare(tag):
    # load the model; from one forward pass: number of layers, token tiling, and where FIT_LAYERS sit in the stack
    spec = REGISTRY[tag]
    mdl = load_model(tag)
    tokz = LOADED.get(tag, {}).get("tokz") or load_tokenizer(tag)
    LOADED[tag] = dict(spec=spec, tokz=tokz)
    POS.pop(tag, None)
    ids, am, pos, spans = encode(tokz, spec, PROBE[:2])
    S = hidden_stack(tag, mdl, ids, am)
    for j, Sj in enumerate(S):
        assert Sj.ndim == 3 and int(pos[j].max()) < Sj.shape[1]
        b = spans[j]
        assert b[0, 0] == 0 and b[-1, 1] == WLEN and (b[1:, 0] == b[:-1, 1]).all(), f"{tag}: tokens do not tile"
    nl = S[0].shape[0]
    lids = list(range(1, nl + 1)) if spec.get("out") == "dnabert2" else list(range(nl))   # DNABERT-2: no embedding layer
    DEPTH[tag] = lids[-1]
    miss = [l for l in FIT_LAYERS[tag] if l not in lids]
    assert not miss, f"{tag}: layers {miss} do not exist (ids {lids[0]}..{lids[-1]})"
    POS[tag] = [lids.index(l) for l in FIT_LAYERS[tag]]
    print(f"{tag}: {nl} layers (depth {DEPTH[tag]}), fitting {FIT_LAYERS[tag]}, hidden {S[0].shape[2]}, "
          f"{len(pos[0])} tokens for {WLEN} bp")
    del S
    return mdl


@torch.no_grad()
def readout(tag, model, seqs, lohi):
    # (n, layers, D): D = hidden, or 2 * hidden for a causal model (forward ‖ reverse complement)
    L = LOADED[tag]
    tokz, spec = L["tokz"], L["spec"]
    passes = [(seqs, lohi)]
    if spec["causal"]:
        passes.append(([rc(s) for s in seqs], [(WLEN - b, WLEN - a) for a, b in lohi]))
    bs = spec.get("batch", BATCH)
    parts = []
    for ss, lh in passes:
        out = None
        for i in range(0, len(ss), bs):
            chunk, lhc = ss[i:i + bs], lh[i:i + bs]
            ids, am, pos, spans = encode(tokz, spec, chunk)
            S = hidden_stack(tag, model, ids, am)
            if out is None:
                out = np.zeros((len(ss), S[0].shape[0], S[0].shape[2]), np.float16)
            for j in range(len(chunk)):
                w = torch.tensor(span_weights(spans[j], *lhc[j]), device=DEV)
                p = torch.tensor(pos[j], device=DEV)
                out[i + j] = (S[j][:, p] * w[None, :, None]).sum(1).half().cpu().numpy()
            del S
        parts.append(out)
    if len(parts) == 1:
        return parts[0]
    if spec.get("rc_mode", "concat") == "mean":
        return ((parts[0].astype(np.float32) + parts[1].astype(np.float32)) / 2).astype(np.float16)
    return np.concatenate(parts, 2)

## 2.2 P(bound) from the all-sites direction, and a check against the out-of-fold P of the pooled hits

P = sigmoid(((x − mu) / sd) · w + b) with `w_all`, `mu`, `sd` of the TF's fit at layer 24. The check: the same
formula on the Kircher hits of 40 TFs, against their out-of-fold `prob_hit` (5-fold mean). They are not the same
quantity, but they should rank the hits almost the same.

In [ ]:
def direction(m):
    z = np.load(f"{FIT_DIR}/{m}.npz", allow_pickle=False)
    li = [int(l) for l in z["layers"]].index(LAYER)
    return z, li, z["w_all"][li].astype(np.float64), z["mu"][li].astype(np.float64), z["sd"][li].astype(np.float64)


def p_bound(m, windows, w_motif, mdl):
    z, li, w, mu, sd = direction(m)
    X = readout(TAG, mdl, windows, [(MOFF, MOFF + w_motif)] * len(windows))[:, 0].astype(np.float64)
    return 1 / (1 + np.exp(-(((X - mu) / sd) @ w[:-1] + w[-1])))


mdl = prepare(TAG)
HBY = {}
for h in MWJ["hits"]:
    HBY.setdefault(h["motif"], []).append(h)
chk = []
for m in sorted(HBY, key=lambda m: -len(HBY[m]))[:40]:
    z, li, *_ = direction(m)
    ref = pd.Series(z["prob_hit"][:, li], index=pd.MultiIndex.from_arrays([z["hit_region"], z["hit_id"]]))
    hs = HBY[m]
    p = p_bound(m, [h["window"] for h in hs], hs[0]["end"] - hs[0]["start"], mdl)
    chk.append(pd.DataFrame({"motif": m, "all sites": p, "out-of-fold": ref.loc[[(h["region"], h["hit_id"]) for h in hs]].values}))
chk = pd.concat(chk)
print(f"{len(chk)} pooled hits of 40 TFs: Spearman(all-sites P, out-of-fold P) = "
      f"{stats.spearmanr(chk['all sites'], chk['out-of-fold'])[0]:.3f}, median |ΔP| = {(chk['all sites'] - chk['out-of-fold']).abs().median():.3f}")

## 2.3 Every occurrence

One file per TF in `malinois/p_nt500m/`, so an interrupted session continues where it stopped.

In [ ]:
todo = [(m, g) for m, g in U.groupby("motif") if not os.path.exists(f"{OUTP}/{m}.csv")]
n_left = sum(len(g) for _, g in todo)
print(f"{len(todo)} TFs, {n_left:,} occurrences to do")
t_all, done = time.time(), 0
for k, (m, g) in enumerate(todo):
    w = int(g["b"].iloc[0] - g["a"].iloc[0])
    wins = [GENOME[c][s - MOFF:s - MOFF + WLEN].seq for c, s in zip(g["chrom38"], g["g38"])]
    assert all(len(x) == WLEN and "N" not in x for x in wins)
    p = p_bound(m, wins, w, mdl)
    pd.DataFrame({"uid": g["uid"].values, "P": p}).to_csv(f"{OUTP}/{m}.part.csv", index=False)
    os.replace(f"{OUTP}/{m}.part.csv", f"{OUTP}/{m}.csv")
    done += len(g)
    el = time.time() - t_all
    print(f"[{k + 1}/{len(todo)}] {m:10s} {len(g):6,} occurrences  mean P {p.mean():.2f}  "
          f"[{el / 60:.1f} min; ~{el / done * (n_left - done) / 60:.0f} min left]", flush=True)
del mdl; gc.collect(); torch.cuda.empty_cache()

---
# Section 3 — evaluation *(any runtime; run 3.1 first on a fresh CPU runtime)*

Per cell (HepG2, K562, SK-N-SH), over the variants with at least one motifbreakR call on a TF that has a P(bound):

* **3.2 Prediction.** motifbreakR (the call with the largest |alleleDiff|) vs motifbreakR + CAV (the call with the
  highest P, alleleDiff × w(P)) against the measured Δ: Pearson and Spearman. Null N2: P shuffled within TF.
* **3.3 Calibration.** For the occurrence the CAV picks: Spearman(P, |Δ|), and the same after removing what
  motifbreakR's own largest |alleleDiff| explains (partial Spearman). Null N2.
* **3.4 Cell contrast.** HepG2 minus SK-N-SH and K562 minus SK-N-SH of those numbers, 95% bootstrap interval over
  variants (the same variants in every cell).
* **3.5 By P.** In fifths of P: mean |Δ| percentile, and how often motifbreakR's direction (sign of alleleDiff) matches
  the sign of Δ among clear effects (|Δ / SE| ≥ 2).

Two sets of variants: **all**, and **active** (the element has log2FC ≥ 1 for at least one allele in that cell; an
element that does nothing cannot lose activity).

## 3.1 Load

In [ ]:
WORK_DIR = "/content/drive/MyDrive/GitHub/Biological-Foundation-Model/Notebooks/EP Pair Evaluation/per_occurrence/motif_token_study"
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

MAL = f"{WORK_DIR}/malinois"
OUT = f"{MAL}/eval_nt500m"
os.makedirs(OUT, exist_ok=True)
CELLS3 = ["HepG2", "K562", "SKNSH"]
MID, K = 0.75, 20
ACTIVE_MIN = 1.0
Z_CLEAR = 2.0
N_PERM, N_BOOT = 1000, 1000
wfun = lambda x: 1 / (1 + np.exp(-K * (x - MID)))
pd.set_option("display.width", 250)

V = pd.read_csv(f"{MAL}/variants.csv.gz", index_col="vi")
H = pd.read_csv(f"{MAL}/hits.csv")
CALLS = pd.read_csv(f"{MAL}/calls.csv")
PD = f"{MAL}/p_nt500m"
PU = pd.concat([pd.read_csv(f"{PD}/{f}") for f in os.listdir(PD) if f.endswith(".csv") and not f.endswith(".part.csv")])
missing = set(H.loc[~H["leak"], "motif"]) - set(H.loc[H["uid"].isin(PU["uid"]), "motif"])
if missing:
    print(f"INCOMPLETE: {len(missing)} TFs have no P yet; their calls are left out")
H = H[~H["leak"]].merge(PU, on="uid")
PVM = H.groupby(["vi", "motif"])["P"].max().rename("P").reset_index()
L = CALLS.merge(PVM, on=["vi", "motif"]).sort_values(["vi", "motif"]).reset_index(drop=True)
VI = np.unique(L["vi"].values)
cv = np.searchsorted(VI, L["vi"].values)                  # call -> variant (0..len(VI)-1)
cm = pd.factorize(L["motif"])[0]
AD, P0 = L["alleleDiff"].values, L["P"].values
NV = len(VI)


def pick(key):
    # per variant, the call with the largest key
    o = np.lexsort((key, cv))
    return o[np.r_[cv[o][1:] != cv[o][:-1], True]]


FB = pick(np.abs(AD))
PRED_MBR, ADMAX = AD[FB], np.abs(AD[FB])


def cav(p):
    j = pick(p + 1e-9 * np.abs(AD))
    return AD[j] * wfun(p[j]), p[j], AD[j]


_base = np.argsort(cm, kind="stable")


def shuffle_tf(p, rng):
    # P permuted among the calls of the same TF
    o = np.lexsort((rng.random(len(p)), cm))
    out = np.empty_like(p)
    out[_base] = p[o]
    return out


Y = {c: V.loc[VI, f"d_{c}"].values for c in CELLS3}
SE = {c: V.loc[VI, f"se_{c}"].values for c in CELLS3}
SETS = {(c, s): np.isfinite(Y[c]) & ((V.loc[VI, f"act_{c}"].values >= ACTIVE_MIN) if s == "active" else True)
        for c in CELLS3 for s in ("all", "active")}
print(f"{NV:,} variants with a motifbreakR call on a TF with P(bound) ({L['motif'].nunique()} TFs, {len(L):,} calls); "
      f"median {np.median(np.bincount(cv)):.0f} call(s) per variant, {np.mean(np.bincount(cv) > 1):.0%} with more than one")
print(pd.Series({f"{c} {s}": int(m.sum()) for (c, s), m in SETS.items()}).to_string())

## 3.2 Prediction: motifbreakR vs motifbreakR + CAV

In [ ]:
def rank(x):
    return stats.rankdata(x)


def pear(a, b):
    a, b = a - a.mean(), b - b.mean()
    return float((a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum()))


def pred_stats(pred, m, c):
    y = Y[c][m]
    return pear(PRED_MBR[m], y), pear(pred[m], y), pear(rank(PRED_MBR[m]), rank(y)), pear(rank(pred[m]), rank(y))


rng = np.random.default_rng(0)
PRED_CAV, PCH, ADCH = cav(P0)
NULL_P = [shuffle_tf(P0, rng) for _ in range(N_PERM)]
NULL_C = [cav(p) for p in NULL_P]
rows = []
for (c, s), m in SETS.items():
    a, b, ra, rb = pred_stats(PRED_CAV, m, c)
    null = np.array([pred_stats(x[0], m, c) for x in NULL_C])
    rows.append({"cell": c, "variants": s, "n": int(m.sum()),
                 "Pearson mbR": a, "Pearson mbR + CAV": b, "gain": b - a, "p N2": np.mean(null[:, 1] >= b),
                 "Spearman mbR": ra, "Spearman mbR + CAV": rb, "gain ": rb - ra, "p N2 ": np.mean(null[:, 3] >= rb)})
T2 = pd.DataFrame(rows).set_index(["cell", "variants"])
T2.to_csv(f"{OUT}/prediction.csv")
print("measured Δ = log2FC(alt) - log2FC(ref); mbR = motifbreakR; p N2 = share of TF-wise shuffles of P at least as good\n")
print(T2.to_string(float_format=lambda x: f"{x:+.3f}"))

## 3.3 Calibration: is the effect larger where the CAV says bound?

In [ ]:
def partial_spearman(x, y, z):
    rx, ry, rz = rank(x), rank(y), rank(z)
    res = lambda a: a - np.polyval(np.polyfit(rz, a, 1), rz)
    return pear(res(rx), res(ry))


def cal_stats(p, m, c):
    y = np.abs(Y[c][m])
    return pear(rank(p[m]), rank(y)), partial_spearman(p[m], y, ADMAX[m])


rows = []
for (c, s), m in SETS.items():
    o = cal_stats(PCH, m, c)
    null = np.array([cal_stats(x[1], m, c) for x in NULL_C])
    rows.append({"cell": c, "variants": s, "n": int(m.sum()),
                 "Spearman(P, |Δ|)": o[0], "p N2": np.mean(null[:, 0] >= o[0]),
                 "after removing |alleleDiff|": o[1], "p N2 ": np.mean(null[:, 1] >= o[1]),
                 "Spearman(|alleleDiff|, |Δ|)": pear(rank(ADMAX[m]), rank(np.abs(Y[c][m])))})
T3 = pd.DataFrame(rows).set_index(["cell", "variants"])
T3.to_csv(f"{OUT}/calibration.csv")
print("P = P(bound) of the occurrence the CAV picks; |Δ| = measured |allelic effect|\n")
print(T3.to_string(float_format=lambda x: f"{x:+.3f}"))

## 3.4 Cell contrast: HepG2 or K562 minus SK-N-SH, on the same variants

In [ ]:
def contrast_stats(idx, c, s):
    out = []
    for cc in (c, "SKNSH"):
        m = idx[SETS[cc, s][idx]]
        a, b, _, _ = pred_stats(PRED_CAV, m, cc)
        out.append([b - a, *cal_stats(PCH, m, cc)])
    return np.array(out[0]) - np.array(out[1])


rng = np.random.default_rng(1)
rows = []
for c in ("HepG2", "K562"):
    for s in ("all", "active"):
        both = np.flatnonzero(SETS[c, s] | SETS["SKNSH", s])
        o = contrast_stats(both, c, s)
        bs = np.array([contrast_stats(rng.choice(both, len(both)), c, s) for _ in range(N_BOOT)])
        lo, hi = np.percentile(bs, [2.5, 97.5], 0)
        for k, nm in enumerate(["gain in Pearson", "Spearman(P, |Δ|)", "after removing |alleleDiff|"]):
            rows.append({"contrast": f"{c} - SK-N-SH", "variants": s, "statistic": nm, "difference": o[k],
                         "95% CI low": lo[k], "95% CI high": hi[k]})
T4 = pd.DataFrame(rows).set_index(["contrast", "variants", "statistic"])
T4.to_csv(f"{OUT}/cell_contrast.csv")
print("positive = the CAV does better in its training cell than in SK-N-SH; bootstrap over the variants of either cell,"
      " each cell using its own (active) variants\n")
print(T4.to_string(float_format=lambda x: f"{x:+.3f}"))

## 3.5 By P(bound): effect size and direction agreement

In [ ]:
q = pd.qcut(PCH, 5, labels=False, duplicates="drop")
rows = []
for c in CELLS3:
    m = SETS[c, "active"]
    pct = np.full(NV, np.nan); pct[m] = 100 * rank(np.abs(Y[c][m])) / m.sum()
    clear = m & (np.abs(Y[c]) / SE[c] >= Z_CLEAR)
    for k in range(q.max() + 1):
        sel = q == k
        rows.append({"cell": c, "P fifth": k + 1, "P range": f"{PCH[sel].min():.2f}-{PCH[sel].max():.2f}",
                     "n active": int((sel & m).sum()), "mean |Δ| percentile": np.nanmean(pct[sel & m]),
                     "n clear": int((sel & clear).sum()),
                     "direction agrees (clear)": np.mean(np.sign(ADCH[sel & clear]) == np.sign(Y[c][sel & clear]))})
T5 = pd.DataFrame(rows).set_index(["cell", "P fifth"])
T5.to_csv(f"{OUT}/by_p_fifth.csv")
print(f"active variants; clear = |Δ / SE| >= {Z_CLEAR}\n")
print(T5.to_string(float_format=lambda x: f"{x:.3f}"))

INK = "#52514e"
COL = {"HepG2": "#2a6f97", "K562": "#c8553d", "SKNSH": "#8a8a8a"}
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for c in CELLS3:
    d = T5.loc[c]
    ax[0].plot(d.index, d["mean |Δ| percentile"], "-o", c=COL[c], label=c, ms=4)
    ax[1].plot(d.index, d["direction agrees (clear)"], "-o", c=COL[c], label=c, ms=4)
ax[0].axhline(50, c=INK, lw=.6, ls=":"); ax[1].axhline(.5, c=INK, lw=.6, ls=":")
ax[0].set_ylabel("mean |Δ| percentile (active variants)"); ax[1].set_ylabel("sign(alleleDiff) = sign(Δ), clear effects")
for a in ax:
    a.set_xlabel("fifth of P(bound) of the CAV's pick"); a.spines[["top", "right"]].set_visible(False)
ax[0].legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig(f"{OUT}/by_p_fifth.png", dpi=150, bbox_inches="tight"); plt.show()

## End

Outputs in `malinois/eval_nt500m/`: `prediction.csv`, `calibration.csv`, `cell_contrast.csv`, `by_p_fifth.csv`,
`by_p_fifth.png`.

In [ ]:
from google.colab import runtime
runtime.unassign()